# First PM++ simulation

{bdg-primary}`2 CUDA GPUs` {bdg-secondary}`256^3 particles` {bdg-success}`saved outputs`

This notebook follows one complete PM++ simulation: configuration, cosmology tables,
white noise, linear modes, 2LPT, evolution to $a=1$, CIC scatter, and density
projections. It uses the same two-device `mesh_halo` ownership, communication, and
sharding contract as the production diagnostics in notebook 04.

```{admonition} Committed execution record
:class: note
All outputs and plots on this page are committed in the notebook; hosted documentation
does not execute them.  The first executed output is the environment and configuration
record for the published run.
```


In [ ]:
%matplotlib inline
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from pmpp.boltzmann import boltzmann
from pmpp.configuration import Configuration
from pmpp.cosmo import SimpleLCDM
from pmpp.lpt import lpt
from pmpp.modes import linear_modes, white_noise
from pmpp.multigpu_configuration import MultiGPUConfiguration
from pmpp.nbody import nbody
from pmpp.scatter import scatter
from pmpp.utils import create_compute_mesh

SEED = 0
RESOLUTION = 256
BOX_SIZE = 100.0  # Mpc/h

selected_devices = [device for device in jax.devices() if device.platform == "gpu"][:2]
if len(selected_devices) != 2:
    raise RuntimeError("This precomputed simulation notebook requires two visible CUDA devices.")
execution_record = {
    "jax_version": jax.__version__,
    "backend": jax.default_backend(),
    "selected_device_count": len(selected_devices),
    "visible_device_count": len(jax.devices()),
    "selected_device_models": [device.device_kind for device in selected_devices],
    "seed": SEED,
    "configuration": {
        "box_size_mpc_h": BOX_SIZE,
        "particle_grid": (RESOLUTION,) * 3,
        "mesh_ratio": 1,
        "lpt_order": 2,
        "a_start": 1 / 64,
        "a_stop": 1.0,
        "a_nbody_maxstep": 1 / 64,
        "mode": "mesh_halo",
    },
    "capacities": {
        "max_ptcl_per_slice": 10_066_329,
        "max_share_ptcl": 1_600_000,
        "max_halo_share_ptcl": 800_000,
        "max_share_gather_ptcl": 1_800_000,
    },
}
print("PM++ DOCUMENTATION EXECUTION RECORD")
for key, value in execution_record.items():
    print(f"{key}: {value}")
compute_mesh = create_compute_mesh(selected_devices)
conf = Configuration(
    ptcl_spacing=BOX_SIZE / RESOLUTION,
    ptcl_grid_shape=(RESOLUTION,) * 3,
    mesh_shape=1,
    multigpu=MultiGPUConfiguration(compute_mesh=compute_mesh, mode="mesh_halo"),
    max_ptcl_per_slice=10_066_329,
    max_share_ptcl=1_600_000,
    max_halo_share_ptcl=800_000,
    max_share_gather_ptcl=1_800_000,
    lpt_order=2,
    a_start=1 / 64,
    a_stop=1.0,
    a_nbody_maxstep=1 / 64,
    float_dtype=jnp.float32,
)

print(f"backend={jax.default_backend()}, devices={[device.device_kind for device in selected_devices]}")
print(f"particles={conf.ptcl_num:,}, mesh={conf.mesh_shape}, box={conf.box_size} Mpc/h")
print(f"scale-factor schedule={np.asarray(conf.a_nbody)}")


## Run the forward model

`boltzmann` tabulates transfer and growth functions.  The remaining calls construct the
initial particle state, evolve it, and deposit every authoritative particle on the mesh.
The configuration is static to JAX, so changing grid sizes, schedules, dtypes, or buffer
capacities triggers recompilation.


In [ ]:
def full_simulation(seed):
    cosmo = boltzmann(SimpleLCDM(conf), conf)
    noise = white_noise(seed, conf)
    modes = linear_modes(noise, cosmo, conf)
    particles = lpt(modes, cosmo, conf)
    particles = nbody(particles, cosmo, conf)
    density = scatter(particles, conf)
    return noise, particles, density

compiled_simulation = jax.jit(full_simulation)
noise, particles_final, density = compiled_simulation(SEED)
density.block_until_ready()

active_particles = int(np.count_nonzero(~np.asarray(particles_final.unused_index)))
summary = {
    "noise_shape": tuple(noise.shape),
    "particle_slots": tuple(particles_final.disp.shape),
    "active_particles": active_particles,
    "density_shape": tuple(density.shape),
}
summary


## Scientific sanity checks

CIC mass assignment preserves total particle weight.  For one particle per Lagrangian
cell, the mesh mean should therefore be one and the mesh sum should equal the number of
active particles (up to floating-point reduction error).


In [ ]:
density_host = np.asarray(density)
checks = {
    "all_finite": bool(np.isfinite(density_host).all()),
    "mean_density": float(density_host.mean()),
    "mesh_mass": float(density_host.sum()),
    "relative_mass_error": float(abs(density_host.sum() - active_particles) / active_particles),
}

assert density.shape == conf.mesh_shape
assert checks["all_finite"]
np.testing.assert_allclose(checks["mean_density"], 1.0, rtol=2e-5, atol=2e-5)
np.testing.assert_allclose(checks["mesh_mass"], active_particles, rtol=2e-5)
checks


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.3), constrained_layout=True)
log_projections = [
    np.log10(np.clip(density_host.sum(axis=axis), 1e-6, None))
    for axis in range(3)
]
all_log_values = np.concatenate([projection.ravel() for projection in log_projections])
vmin, vmax = np.percentile(all_log_values, (0.5, 99.5))
for axis, (ax, log_projection) in enumerate(zip(axes, log_projections)):
    image = ax.imshow(log_projection, origin="lower", cmap="magma", vmin=vmin, vmax=vmax)
    ax.set_title(f"log10 sum along axis {axis}")
    ax.set_xlabel("mesh cell")
    ax.set_ylabel("mesh cell")
    fig.colorbar(image, ax=ax, shrink=0.78, label="log10 projected density")
plt.show()


## Where to go next

- Notebook 02 explains configuration and recompilation boundaries.
- Notebook 03 demonstrates resolution-consistent nested noise through $z=0$.
- Notebook 04 is the recommended two-GPU `mesh_halo` PM++ path.
- Notebook 05 records observer frames and computes a differentiable power spectrum.
